# ⚖️ Handling Imbalanced Data
**One-line description:** Tackle the accuracy paradox and build models that actually detect rare events like fraud, disease, or failures.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/06_imbalanced_data.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install imbalanced-learn scikit-learn pandas numpy matplotlib seaborn --quiet


## 📖 What is Imbalanced Data?

Imbalanced data occurs when the classes in your target variable are **not represented equally** — one class (majority) vastly outnumbers another (minority).

**The Accuracy Paradox:** If only 1% of transactions are fraudulent, a model that predicts "Not Fraud" for every single transaction achieves **99% accuracy** — but it never detects a single fraud case!

**Real-world imbalanced problems:**
| Domain | Problem | Typical Imbalance |
|--------|---------|-----------------|
| Finance | Fraud detection | 0.1% - 2% fraud |
| Medicine | Rare disease diagnosis | 1% - 5% positive |
| Manufacturing | Defect detection | 0.01% - 1% defects |
| Security | Intrusion detection | < 1% attacks |
| Insurance | Claim fraud | 1% - 3% fraud |


## 💡 Why Does It Matter?

- Standard models optimize **overall accuracy** — which ignores minority classes
- **False negatives** on fraud/disease are catastrophic in real applications
- You need **special metrics** (F1, PR-AUC) to actually measure performance
- You need **special techniques** to help models learn minority patterns


## ⚙️ How Does It Work?

We'll cover a complete toolkit:
1. **The Accuracy Paradox** — demonstrate why accuracy is misleading
2. **Class Weights** — tell the model minority class errors cost more
3. **Undersampling** — remove majority samples (RandomUnderSampler, TomekLinks)
4. **Oversampling** — add minority samples (SMOTE, ADASYN)
5. **Combination** — SMOTETomek
6. **Proper Metrics** — F1, Precision, Recall, ROC-AUC, PR-AUC


## 🛠️ Hands-on Code

### Step 1: Create a Fraud Detection Dataset (1% Fraud Rate)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score,
                              precision_recall_curve, roc_curve,
                              f1_score, precision_score, recall_score)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Create a highly imbalanced fraud detection dataset
X, y = make_classification(
    n_samples=10000,
    n_features=15,
    n_informative=10,
    n_redundant=2,
    weights=[0.99, 0.01],  # 99% legitimate, 1% fraud
    random_state=42,
    flip_y=0.001
)

feature_names = [
    'TransactionAmt', 'MerchantRiskScore', 'UserDeviceAge',
    'NumPrevDeclines', 'AvgTxPerDay', 'LocationEntropy',
    'TimeSinceLastTx', 'AmtVsAvgRatio', 'CrossBorderFlag',
    'NewMerchantFlag', 'IPRiskScore', 'CardAge',
    'PhoneChangeRecent', 'EmailDomainRisk', 'AccountBalance'
]

df = pd.DataFrame(X, columns=feature_names)
df['Fraud'] = y

print("Fraud Detection Dataset:")
print(f"  Total transactions: {len(df):,}")
print(f"  Legitimate: {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"  Fraud:      {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")


In [ ]:
# --- Visualization 1: Class imbalance and the accuracy paradox ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Class distribution
counts = df['Fraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'red'], edgecolor='black')
axes[0].set_title('Extreme Class Imbalance
(1% Fraud Rate)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (bar, v) in enumerate(zip(axes[0].patches, counts.values)):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Accuracy paradox demonstration
strategies = ['Predict All\nLegitimate', 'Random\nPredict', 'Perfect\nModel']
accuracy   = [0.99, 0.50, 1.00]
fraud_recall = [0.00, 0.50, 1.00]

x = np.arange(len(strategies))
width = 0.35
bars1 = axes[1].bar(x - width/2, accuracy, width, label='Accuracy', color='steelblue', alpha=0.8)
bars2 = axes[1].bar(x + width/2, fraud_recall, width, label='Fraud Recall', color='red', alpha=0.8)
axes[1].set_title('The Accuracy Paradox', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strategies, fontsize=9)
axes[1].legend()
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)

# Feature distribution by class
axes[2].hist(df[df['Fraud']==0]['TransactionAmt'], bins=50, alpha=0.6,
             color='steelblue', label='Legitimate', density=True)
axes[2].hist(df[df['Fraud']==1]['TransactionAmt'], bins=50, alpha=0.6,
             color='red', label='Fraud', density=True)
axes[2].set_title('TransactionAmt Distribution\nby Class', fontsize=12, fontweight='bold')
axes[2].set_xlabel('TransactionAmt')
axes[2].set_ylabel('Density')
axes[2].legend()

plt.suptitle('Fraud Detection: Understanding Class Imbalance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fraud_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 2: Train/Test Split and Baseline Model


In [ ]:
# Stratified split to preserve 1% fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_names].values, df['Fraud'].values,
    test_size=0.2, random_state=42, stratify=df['Fraud'].values
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate_model(model_name, y_true, y_pred, y_prob):
    # Calculate comprehensive metrics for imbalanced classification
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Model': model_name,
        'Accuracy':  round(((y_pred == y_true).mean()), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1 Score':  round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_prob), 4),
        'PR-AUC':    round(average_precision_score(y_true, y_prob), 4),
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn
    }

results = []

# Baseline: Logistic Regression with no balancing
lr_base = LogisticRegression(random_state=42, max_iter=1000)
lr_base.fit(X_train_scaled, y_train)
y_pred_base = lr_base.predict(X_test_scaled)
y_prob_base = lr_base.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('Baseline LR', y_test, y_pred_base, y_prob_base))
print("Baseline model trained. Metrics:")
print(f"  Accuracy: {results[-1]['Accuracy']:.4f}  ← looks great, but...")
print(f"  Recall (Fraud detection rate): {results[-1]['Recall']:.4f}  ← terrible!")
print(f"  Only catching {results[-1]['TP']} out of {results[-1]['TP']+results[-1]['FN']} frauds!")


### Step 3: Class Weights Adjustment


In [ ]:
# --- Class Weights: tell the model to penalize fraud misclassification more ---
# class_weight='balanced' automatically sets weights inversely proportional to frequency

lr_weighted = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_weighted.fit(X_train_scaled, y_train)
y_pred_w = lr_weighted.predict(X_test_scaled)
y_prob_w = lr_weighted.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('LR + Class Weights', y_test, y_pred_w, y_prob_w))

print("Class Weight Adjustment:")
print(f"  Recall improved: {results[0]['Recall']:.4f} → {results[-1]['Recall']:.4f}")
print(f"  F1 Score:        {results[0]['F1 Score']:.4f} → {results[-1]['F1 Score']:.4f}")
print("\nNote: accuracy may decrease as model now makes more false positives to catch fraud")


### Step 4: Undersampling, Oversampling, and Combined Methods


In [ ]:
# --- Random Undersampling ---
rus = RandomUnderSampler(sampling_strategy=0.3, random_state=42)  # 30% minority ratio
X_rus, y_rus = rus.fit_resample(X_train_scaled, y_train)
lr_rus = LogisticRegression(random_state=42, max_iter=1000)
lr_rus.fit(X_rus, y_rus)
y_pred_rus = lr_rus.predict(X_test_scaled)
y_prob_rus = lr_rus.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('Random Undersampling', y_test, y_pred_rus, y_prob_rus))
print(f"After RandomUnderSampler: {pd.Series(y_rus).value_counts().to_dict()}")

# --- TomekLinks Undersampling ---
tl = TomekLinks(sampling_strategy='majority')
X_tl, y_tl = tl.fit_resample(X_train_scaled, y_train)
lr_tl = LogisticRegression(random_state=42, max_iter=1000)
lr_tl.fit(X_tl, y_tl)
y_pred_tl = lr_tl.predict(X_test_scaled)
y_prob_tl = lr_tl.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('TomekLinks', y_test, y_pred_tl, y_prob_tl))
print(f"After TomekLinks: {pd.Series(y_tl).value_counts().to_dict()}")

# --- SMOTE Oversampling ---
smote = SMOTE(sampling_strategy=0.5, random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train)
lr_smote = LogisticRegression(random_state=42, max_iter=1000)
lr_smote.fit(X_smote, y_smote)
y_pred_smote = lr_smote.predict(X_test_scaled)
y_prob_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('SMOTE', y_test, y_pred_smote, y_prob_smote))
print(f"\nAfter SMOTE: {pd.Series(y_smote).value_counts().to_dict()}")

# --- ADASYN Oversampling ---
adasyn = ADASYN(sampling_strategy=0.5, random_state=42, n_neighbors=5)
X_adasyn, y_adasyn = adasyn.fit_resample(X_train_scaled, y_train)
lr_adasyn = LogisticRegression(random_state=42, max_iter=1000)
lr_adasyn.fit(X_adasyn, y_adasyn)
y_pred_adasyn = lr_adasyn.predict(X_test_scaled)
y_prob_adasyn = lr_adasyn.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('ADASYN', y_test, y_pred_adasyn, y_prob_adasyn))
print(f"After ADASYN: {pd.Series(y_adasyn).value_counts().to_dict()}")

# --- SMOTETomek Combination ---
smotetomek = SMOTETomek(sampling_strategy=0.5, random_state=42)
X_smt, y_smt = smotetomek.fit_resample(X_train_scaled, y_train)
lr_smt = LogisticRegression(random_state=42, max_iter=1000)
lr_smt.fit(X_smt, y_smt)
y_pred_smt = lr_smt.predict(X_test_scaled)
y_prob_smt = lr_smt.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model('SMOTETomek', y_test, y_pred_smt, y_prob_smt))
print(f"After SMOTETomek: {pd.Series(y_smt).value_counts().to_dict()}")


In [ ]:
# --- Comprehensive results comparison ---
results_df = pd.DataFrame(results)
print("\n" + "="*90)
print("COMPREHENSIVE METRICS COMPARISON")
print("="*90)
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'PR-AUC', 'TP', 'FN']
print(results_df[display_cols].to_string(index=False))
print("\nTP = True Positives (fraud correctly caught)")
print("FN = False Negatives (fraud missed — the costly mistake!)")


In [ ]:
# --- Visualization 2: Multi-metric comparison and PR/ROC curves ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Metric comparison bar chart
metrics_to_plot = ['Recall', 'F1 Score', 'PR-AUC', 'ROC-AUC']
models = results_df['Model'].tolist()
x = np.arange(len(models))
width = 0.2
colors_m = ['coral', 'steelblue', 'green', 'purple']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_m)):
    axes[0, 0].bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.8)

axes[0, 0].set_xticks(x + width * 1.5)
axes[0, 0].set_xticklabels(models, rotation=30, ha='right', fontsize=8)
axes[0, 0].set_title('Multi-Metric Comparison Across Strategies', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].legend(fontsize=9)
axes[0, 0].set_ylim(0, 1.1)

# Fraud caught vs missed
axes[0, 1].bar(models, results_df['TP'], color='green', edgecolor='black', label='Fraud Caught (TP)')
axes[0, 1].bar(models, results_df['FN'], bottom=results_df['TP'], color='red',
               edgecolor='black', alpha=0.7, label='Fraud Missed (FN)')
axes[0, 1].set_xticklabels(models, rotation=30, ha='right', fontsize=8)
axes[0, 1].set_title('Fraud Cases: Caught vs Missed', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()

# PR Curves
models_probs = {
    'Baseline': y_prob_base,
    'Class Weights': y_prob_w,
    'SMOTE': y_prob_smote,
    'SMOTETomek': y_prob_smt,
}
for name, probs in models_probs.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    axes[1, 0].plot(rec, prec, label=f'{name} (AP={ap:.3f})', linewidth=2)

axes[1, 0].axhline(y=y_test.mean(), color='black', linestyle='--', label=f'Random (AP={y_test.mean():.3f})')
axes[1, 0].set_title('Precision-Recall Curves
(higher AUC = better for imbalanced data)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Recall (Fraud detection rate)')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend(fontsize=9)

# ROC Curves
for name, probs in models_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[1, 1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[1, 1].plot([0,1], [0,1], 'k--', label='Random (AUC=0.5)')
axes[1, 1].set_title('ROC Curves', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate (Recall)')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Imbalanced Data: Complete Strategy Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalanced_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# --- Confusion Matrix Heatmaps ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

models_for_cm = [
    ('Baseline LR', y_pred_base),
    ('LR + Class Weights', y_pred_w),
    ('Random Undersampling', y_pred_rus),
    ('SMOTE', y_pred_smote),
    ('ADASYN', y_pred_adasyn),
    ('SMOTETomek', y_pred_smt),
]

for ax, (name, preds) in zip(axes.flat, models_for_cm):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'])
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    ax.set_title(f'{name}\nRecall={recall:.3f}, F1={f1:.3f}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices: All Strategies Compared', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()


## 📌 Key Takeaways

- **Never use accuracy alone** for imbalanced data — use F1, Recall, PR-AUC
- **The accuracy paradox**: 99% accuracy on 1% fraud dataset means predicting all "legitimate"
- **Class weights** (`class_weight='balanced'`) is the simplest, safest starting point
- **Undersampling** (RandomUnderSampler) loses majority data — fast but can hurt performance
- **TomekLinks** removes borderline majority samples — gentle cleaning, not full balancing
- **SMOTE** creates synthetic minority samples — effective for most cases
- **ADASYN** adapts to decision boundary difficulty — often outperforms SMOTE
- **SMOTETomek** combines SMOTE + Tomek cleaning — generally best overall
- Use **PR-AUC** as your primary metric for highly imbalanced datasets (not ROC-AUC)
- **Always evaluate on the original imbalanced test set** — augmentation only for training!
